# MIBI Analysis Package Example

This notebook demonstrates how to use the MIBI Analysis package for preprocessing, running MuVI factor analysis, and visualizing results following the patterns from MOFACell.ipynb.

In [ ]:
# Import required packages following MOFACell.ipynb patterns
import mibi_analysis as ma
import muon as mu
import liana as li
import numpy as np
import pandas as pd
from plotnine import *

# If running from the repository, you may need to add the package to path
import sys
sys.path.insert(0, '..')

## 1. Device Detection

Check for available GPU devices as in MOFACell.ipynb.

In [ ]:
# Get available device (GPU if available, CPU otherwise)
device = ma.get_device()
print(f"Using device: {device}")

## 2. Load Data

Load your MuData object containing multicellular features.

In [ ]:
# Load MuData object (following MOFACell.ipynb pattern)
# Replace with your actual data path
features = mu.read_h5mu("../data/celltype_features_with_functional_markers.h5mu")

print(f"Loaded MuData with {len(features.mod)} modalities:")
for modality_name, modality_data in features.mod.items():
    print(f"  {modality_name}: {modality_data.shape[0]} samples, {modality_data.shape[1]} features")

## 3. Data Preprocessing

Normalize features and prepare data for factor analysis following MOFACell.ipynb patterns.

In [ ]:
# Get summary statistics before preprocessing
stats_before = ma.get_modality_stats(features)
print("Statistics before preprocessing:")
print(stats_before)

In [ ]:
# Prepare data for MOFA analysis using the same approach as MOFACell.ipynb
# Center and scale each column of the data, ignoring NA values
ma.prepare_mudata_for_mofa(
    features,
    normalize=True,
    filter_features=True,
    min_variance=0.0,  # Use 0.0 to match notebook behavior
    copy=False  # Modify in place
)

print("Data preprocessing completed.")

In [ ]:
# Check statistics after preprocessing
stats_after = ma.get_modality_stats(features)
print("Statistics after preprocessing:")
print(stats_after)

## 4. Run MOFA Factor Analysis

Run MOFA analysis on the preprocessed data using the same parameters as MOFACell.ipynb.

In [ ]:
# Run MOFA analysis with parameters from MOFACell.ipynb
model_path = ma.run_mofa_analysis(
    features,
    n_factors=10,  # Default from notebook
    outfile='example_mofa_model.h5ad',
    use_obs='union',
    convergence_mode='medium',
    scale_groups=False,
    scale_views=False,
    seed=1337,
    use_var=None
)

print(f"MOFA model saved to: {model_path}")

## 5. Evaluate Model Performance

Calculate and visualize model performance metrics.

In [ ]:
# Calculate R² (variance explained) as in MOFACell.ipynb
r2_results = ma.calculate_model_r2(model_path)
print(f"Overall model R²: {r2_results['mean_r2']:.3f}")
print("\nR² by modality:")
for key, value in r2_results.items():
    if key.endswith('_r2') and key != 'mean_r2':
        modality = key.replace('_r2', '')
        print(f"  {modality}: {value:.3f}")

In [ ]:
# Calculate reconstruction R²
recon_r2 = ma.calculate_reconstruction_r2(model_path, features)
print(f"Macro reconstruction R²: {recon_r2['macro_reconstruction_r2']:.3f}")

In [ ]:
# Plot R² explained
fig = ma.plot_r2_explained(r2_results, title='Variance Explained by Modality')
fig.show()

## 6. Extract Factor Scores

Extract factor scores using liana utilities as in MOFACell.ipynb.

In [ ]:
# Extract factor scores using liana utilities (as in MOFACell.ipynb)
factor_scores = ma.extract_factor_scores(
    features, 
    obsm_key='X_mofa', 
    obs_keys=['Stage'],
    use_liana=True
)

print(f"Factor scores shape: {factor_scores.shape}")
print(f"Available columns: {list(factor_scores.columns)}")
print("\nFirst 5 rows:")
print(factor_scores.head())

## 7. Extract Variable Loadings

Extract variable loadings using liana utilities as in MOFACell.ipynb.

In [ ]:
# Extract variable loadings using liana utilities (as in MOFACell.ipynb)
variable_loadings = ma.extract_factor_loadings(
    features,
    varm_key='LFs',  # 'LFs' for MOFA loadings
    use_liana=True
)

print(f"Variable loadings shape: {variable_loadings.shape}")
print(f"Available columns: {list(variable_loadings.columns)}")
print("\nFirst 5 rows:")
print(variable_loadings.head())

## 8. Test Factor-Clinical Associations

Test for statistical associations between factors and clinical variables.

In [ ]:
# Define clinical variables to test (adjust based on your data)
clinical_vars = ['Stage']  # Add other clinical variables as available

# Filter to only include available clinical variables
available_clinical_vars = [var for var in clinical_vars if var in factor_scores.columns]

if available_clinical_vars:
    # Test associations using Kruskal-Wallis as in MOFACell.ipynb
    associations = ma.test_factor_associations(
        factor_scores,
        clinical_variables=available_clinical_vars,
        test_type='kruskal',  # Use Kruskal-Wallis as in notebook
        correction_method='bonferroni'
    )
    
    print("Factor-clinical associations:")
    print(associations)
    
    # Show significant associations
    significant = associations[associations['significant']]
    if len(significant) > 0:
        print("\nSignificant associations:")
        print(significant[['factor', 'clinical_variable', 'p_adjusted', 'effect_size']])
    else:
        print("\nNo significant associations found.")
else:
    print("No clinical variables available for testing.")

## 9. Visualize Factor Scores

Create scatter plots and other visualizations of factor scores using plotnine.

In [ ]:
# Basic factor score plot using plotnine (as in MOFACell.ipynb)
plot = ma.plot_factor_scores(
    factor_scores,
    x_factor='Factor1',  # Note: liana uses 'Factor1' not 'Factor 1' 
    y_factor='Factor2',
    title='Factor 1 vs Factor 2'
)
print(plot)

In [ ]:
# Factor score plot colored by clinical variable with confidence ellipses
if 'Stage' in factor_scores.columns:
    plot_colored = ma.plot_factor_scores(
        factor_scores,
        x_factor='Factor1',
        y_factor='Factor2',
        color_by='Stage',
        add_ellipses=True,  # Add confidence ellipses as in MOFACell.ipynb
        title='Factor 1 vs Factor 2 by Stage'
    )
    print(plot_colored)
else:
    print("Stage variable not available for coloring.")

In [ ]:
# Factor comparison plots (violin plots as in MOFACell.ipynb)
if 'Stage' in factor_scores.columns:
    fig = ma.plot_factor_comparison(
        factor_scores,
        factor1='Factor1',
        factor2='Factor2',
        group_by='Stage',
        plot_type='violin'
    )
    fig.show()
else:
    print("Stage variable not available for comparison plots.")

## 10. Analyze Factor Loadings

Visualize factor loadings to understand which features contribute to each factor.

In [ ]:
# Plot factor loadings (this will work if we have the loadings as dict)
# Note: with liana, we get a DataFrame, so we need to adapt the visualization
if isinstance(variable_loadings, pd.DataFrame):
    print("Variable loadings extracted using liana:")
    print(f"Shape: {variable_loadings.shape}")
    print(f"Views available: {variable_loadings['view'].unique() if 'view' in variable_loadings.columns else 'No view column'}")
    
    # Show top loadings for first factor
    if 'Factor1' in variable_loadings.columns:
        top_loadings = variable_loadings.nlargest(10, 'Factor1')[['Factor1', 'view']]
        print("\nTop 10 loadings for Factor 1:")
        print(top_loadings)
else:
    # If we have dict format, use the original plotting function
    fig = ma.plot_factor_loadings(
        variable_loadings,
        factors_to_plot=['Factor 1', 'Factor 2', 'Factor 3'],
        title='Factor Loadings Heatmap'
    )
    fig.show()

## 11. Summary Statistics

Generate summary statistics for factors by clinical groups.

In [ ]:
# Summarize factors by clinical groups
if 'Stage' in factor_scores.columns:
    summary = ma.summarize_factor_by_group(
        factor_scores,
        factor='Factor1',  # Use Factor1 (liana naming)
        group_by='Stage'
    )
    print("Factor 1 summary by Stage:")
    print(summary)
else:
    print("Stage variable not available for summary statistics.")

## 12. Save Results

Save factor scores and analysis results for further use.

In [ ]:
# Save factor scores
factor_scores.to_csv('factor_scores_with_metadata.csv', index=True)
print("Factor scores saved to 'factor_scores_with_metadata.csv'")

# Save variable loadings 
if isinstance(variable_loadings, pd.DataFrame):
    variable_loadings.to_csv('variable_loadings.csv', index=True)
    print("Variable loadings saved to 'variable_loadings.csv'")

# Save association results if available
if 'associations' in locals():
    associations.to_csv('factor_clinical_associations.csv', index=False)
    print("Association results saved to 'factor_clinical_associations.csv'")

print("\nAnalysis complete! This workflow follows the same patterns as MOFACell.ipynb.")